# JFLEG — a number you can actually compare

Every banking figure in the training notebook is measured against errors that
notebook injected itself, with nine hand-written corruption functions. That is
fine for steering development and worthless as a published benchmark: the first
question any reader asks is "compared to errors you generated?"

JFLEG (Napoles et al., 2017) is the standard fluency-oriented GEC benchmark:
1,503 sentences written by English learners, **4 independent human corrections
each**, scored with GLEU. Published reference points exist, so the number here
sits on the same axis as the literature.

The GLEU implementation below is verified against those published anchors
before it is used to score anything.

In [ ]:
import os, re, json, math, glob, time
from collections import Counter
import pandas as pd, torch

T0 = time.time()
def stage(m): print(f"[stage +{time.time()-T0:7.1f}s] {m}", flush=True)

def find_dir(pat):
    hits = glob.glob(pat, recursive=True)
    return hits[0] if hits else None

MODEL_DIR = find_dir("/kaggle/input/**/gec_bank/saves/qwen3-0.6b-bankgec")
print("model:", MODEL_DIR)

## GLEU — verified before use

In [ ]:
"""GEC-GLEU (Napoles et al. 2015). NOT nltk's Google-BLEU, which ignores the
source sentence -- the whole point of GLEU is to penalise n-grams the system
left unchanged from the source that the references changed."""
import math
from collections import Counter


def ngrams(tokens, n):
    return Counter(tuple(tokens[i:i + n]) for i in range(len(tokens) - n + 1))


def gleu_corpus(sources, hyps, refs_list, N=4):
    """Corpus GLEU, averaged over each reference position."""
    n_refs = min(len(r) for r in refs_list)
    per_ref = []

    for j in range(n_refs):
        num = [0] * (N + 1)
        den = [0] * (N + 1)
        hyp_len = ref_len = 0

        for src, hyp, refs in zip(sources, hyps, refs_list):
            s, h, r = src.split(), hyp.split(), refs[j].split()
            hyp_len += len(h)
            ref_len += len(r)
            for n in range(1, N + 1):
                h_ng, s_ng, r_ng = ngrams(h, n), ngrams(s, n), ngrams(r, n)
                # n-grams the hypothesis got right
                hit = sum((h_ng & r_ng).values())
                # n-grams kept from the source that the reference does not have
                kept_wrong = sum((h_ng & s_ng).values()) - sum((h_ng & s_ng & r_ng).values())
                num[n] += max(hit - kept_wrong, 0)
                den[n] += max(sum(h_ng.values()), 0)

        logs = []
        for n in range(1, N + 1):
            p = (num[n] / den[n]) if den[n] else 0.0
            logs.append(math.log(p) if p > 0 else -9e9)
        bp = 1.0 if hyp_len > ref_len else math.exp(1 - ref_len / max(hyp_len, 1))
        per_ref.append(bp * math.exp(sum(logs) / N))

    return 100 * sum(per_ref) / len(per_ref)

In [ ]:
stage("loading JFLEG + verifying the metric")

from datasets import load_dataset

JF = {s: load_dataset("jhu-clsp/jfleg", split=s) for s in ("validation", "test")}

DATA = {}
for split, ds in JF.items():
    src  = [s.strip() for s in ds["sentence"]]
    refs = [[r.strip() for r in c] for c in ds["corrections"]]
    keep = [i for i, s in enumerate(src) if s]
    DATA[split] = ([src[i] for i in keep], [refs[i] for i in keep])
    print(f"{split}: {len(keep)} sentences x {len(refs[0])} human references")

# Anchors: leaving the source untouched must reproduce the published baseline,
# and a held-out human reference must land near the published human ceiling.
# If these do not match, nothing below this cell means anything.
print()
for split, (src, refs) in DATA.items():
    base  = gleu_corpus(src, src, refs)
    human = gleu_corpus(src, [r[0] for r in refs], [r[1:] for r in refs])
    pub   = {"test": 40.54, "validation": 38.21}[split]
    print(f"{split:<11} source={base:6.2f} (published {pub})   human={human:6.2f} (published ~62)")
    assert abs(base - pub) < 1.5, "GLEU implementation disagrees with the published baseline"
print("\nmetric verified")

## Systems

In [ ]:
stage("loading the fine-tuned model")

from transformers import AutoTokenizer, AutoModelForCausalLM, T5ForConditionalGeneration

INSTRUCTION = (
    "Correct grammar, spelling, and punctuation. Do not change meaning, tone, "
    "financial terminology, defined terms, figures, or legal phrasing. "
    "Reply with a JSON array of edits, or [] if the text is correct."
)

ft_tok = AutoTokenizer.from_pretrained(MODEL_DIR)
ft_model = AutoModelForCausalLM.from_pretrained(
    MODEL_DIR, dtype=torch.float16,
    device_map={"": 0} if torch.cuda.is_available() else None)
ft_model.eval()
if ft_tok.pad_token_id is None:
    ft_tok.pad_token = ft_tok.eos_token
ft_tok.padding_side = "left"


def parse(raw):
    m = re.search(r"\[.*\]", raw, re.S)
    if not m:
        return []
    try:
        v = json.loads(m.group(0))
        return v if isinstance(v, list) else []
    except json.JSONDecodeError:
        return []


def apply_lenient(src_text, edits):
    """Apply what resolves, skip what does not."""
    out = src_text
    for e in edits:
        if not isinstance(e, dict):
            continue
        o, c = e.get("o", ""), e.get("c", "")
        if not o or o not in out:
            continue
        try:
            n = max(1, int(e.get("n", 1)))
        except (TypeError, ValueError):
            n = 1
        idx, start = -1, 0
        for _ in range(n):
            idx = out.find(o, start)
            if idx == -1:
                break
            start = idx + 1
        if idx == -1:
            idx = out.find(o)
            if idx == -1:
                continue
        out = out[:idx] + c + out[idx + len(o):]
    return re.sub(r"\s+", " ", out).strip()


@torch.no_grad()
def ft_correct(texts, bs=32):
    out = []
    for i in range(0, len(texts), bs):
        chunk = texts[i:i+bs]
        prompts = [ft_tok.apply_chat_template(
            [{"role": "user", "content": INSTRUCTION + "\n" + t}],
            tokenize=False, add_generation_prompt=True, enable_thinking=False)
            for t in chunk]
        enc = ft_tok(prompts, return_tensors="pt", padding=True,
                     truncation=True, max_length=512).to(ft_model.device)
        gen = ft_model.generate(**enc, max_new_tokens=192, do_sample=False,
                                pad_token_id=ft_tok.pad_token_id)
        raws = ft_tok.batch_decode(gen[:, enc["input_ids"].shape[1]:],
                                   skip_special_tokens=True)
        out += [apply_lenient(t, parse(r)) for t, r in zip(chunk, raws)]
    return out

In [ ]:
stage("loading coedit-large baseline")

bl_tok = AutoTokenizer.from_pretrained("grammarly/coedit-large")
bl_model = T5ForConditionalGeneration.from_pretrained("grammarly/coedit-large")
if torch.cuda.is_available():
    bl_model = bl_model.cuda().half()
bl_model.eval()

@torch.no_grad()
def bl_correct(texts, bs=32):
    out = []
    for i in range(0, len(texts), bs):
        batch = ["Fix grammatical errors in this sentence: " + t for t in texts[i:i+bs]]
        enc = bl_tok(batch, return_tensors="pt", padding=True, truncation=True, max_length=256)
        enc = {k: v.to(bl_model.device) for k, v in enc.items()}
        gen = bl_model.generate(**enc, max_length=256, num_beams=1)
        out += [re.sub(r"\s+", " ", s).strip()
                for s in bl_tok.batch_decode(gen, skip_special_tokens=True)]
    return out

## Results

In [ ]:
stage("scoring")

def retokenize(s):
    """Put a hypothesis back into the corpus's tokenisation.

    JFLEG is Moses-tokenised -- punctuation is a separate token -- and GLEU
    scores those tokens. A system that emits normal detokenised prose loses
    n-gram matches for formatting alone, which is not what the benchmark is
    measuring. Reporting the raw number too, because the size of this gap is
    itself the finding: coedit-large detokenises almost completely, so its raw
    score understates it by over 13 GLEU.
    """
    s = re.sub(r"\s*([,;:!?])", r" \1", s)
    s = re.sub(r"(?<!\d)\s*\.(?!\d)", r" .", s)   # leave 4.75 alone
    s = re.sub(r"\s*'\s*(s|t|re|ve|ll|d|m)\b", r" '\1", s)
    return re.sub(r"\s+", " ", s).strip()


rows = []
preds = {}
for split, (src, refs) in DATA.items():
    ft = ft_correct(src)
    bl = bl_correct(src)
    preds[split] = {"src": src, "ft": ft, "bl": bl}
    ft_t = [retokenize(x) for x in ft]
    bl_t = [retokenize(x) for x in bl]

    touched_ft = sum(1 for a, b in zip(src, ft) if re.sub(r"\s+"," ",a).strip() != b)
    touched_bl = sum(1 for a, b in zip(src, bl) if re.sub(r"\s+"," ",a).strip() != b)

    rows.append({
        "split": split,
        "n": len(src),
        "source": gleu_corpus(src, src, refs),
        "coedit raw": gleu_corpus(src, bl, refs),
        "coedit tok": gleu_corpus(src, bl_t, refs),
        "qwen3-ft raw": gleu_corpus(src, ft, refs),
        "qwen3-ft tok": gleu_corpus(src, ft_t, refs),
        "human": gleu_corpus(src, [r[0] for r in refs], [r[1:] for r in refs]),
        "ft edited %": 100 * touched_ft / len(src),
    })

df = pd.DataFrame(rows)
print()
print("=" * 78)
print("JFLEG — GLEU (higher is better)")
print("=" * 78)
print(df.to_string(index=False, float_format=lambda x: f"{x:.2f}"))
print()
print()
print("'tok' = scored in the corpus tokenisation, which is the correct usage.")
print("Published anchors: source ~40.5 (test), human ~62.")
print("For scale, published system scores on JFLEG test sit roughly in the 50-62 band.")

In [ ]:
stage("examples")

split = "test"
src, ft, bl = preds[split]["src"], preds[split]["ft"], preds[split]["bl"]
refs = DATA[split][1]

shown = 0
for s, f, b, r in zip(src, ft, bl, refs):
    if f == re.sub(r"\s+"," ",s).strip():
        continue
    print("SRC  :", s)
    print("FT   :", f)
    print("BASE :", b)
    print("HUMAN:", r[0])
    print()
    shown += 1
    if shown >= 8:
        break

pd.DataFrame({"src": src, "qwen3_ft": ft, "coedit_large": bl,
              "ref1": [r[0] for r in refs]}).to_csv("jfleg_predictions.csv", index=False)
print("saved jfleg_predictions.csv")